# Lab 10: Final project, a shop lakehouse with Delta Lake

**Time:** about 4 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>2.x File formats</b></span><br><span style="color:#d93025"><b>3.x Joins and shuffle</b></span><br><span style="color:#d93025"><b>4.x Data skew</b></span><br><span style="color:#d93025"><b>6.x Best practices</b></span><br><span style="color:#d93025"><b>7.x Delta Lake</b></span>

**The project.** You are the data engineer of the shop. Order files arrive once a month, with mistakes in them. Build a small **lakehouse** with Delta Lake that turns them into a clean table and a daily report, and keeps the report up to date when the next month arrives:

```
data/monthly/*.parquet ──► bronze (raw, append-only) ──► silver (clean, unique orders) ──► gold (daily report per country)
                                                           └──► quarantine (rejected rows + reason)
```

There is no skeleton: write every function yourself, in the empty cells. Each milestone lists the exact rules; the check cell under it inspects your tables. Use what Labs 01 to 09 taught you and their documentation. Part 2 is a timed practice set.

**The monthly files** (`data/monthly/orders_2024-01.parquet` to `-03`) have the same columns as *orders*, but contain on purpose: a few orders that belong to another month, orders with a *quantity* of 0, orders with a negative *amount*, and exact duplicate rows.

## Data information
A small online shop. The setup cell below creates three tables in `data/` (seeded, so everyone gets the same rows):

**orders** (`data/orders.parquet`, 1,000,000 rows, January to March 2024)
* *order_id*, *customer_id* (empty for guest checkout), *product_id*, *quantity* (1 to 5)
* *amount*: quantity × product price, in dollars
* *status*: paid, shipped, cancelled or returned; *payment_method*: card, cash or wallet
* *order_ts*, *order_date*: when the order was placed
* *tracking_code*: a 32-character code

**customers** (`data/customers.csv`, 10,000 rows): *customer_id*, *name*, *country*, *signup_date*, *segment* (retail or business). Customer 1 is "Big Reseller Ltd", who places 30% of all orders.

**products** (`data/products.csv`, 200 rows): *product_id*, *name*, *category*, *price*

## How to work
* Write your answer in the code cell under each question, then run the check cell below it: ✅ correct, ❌ try again.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then the reference solution.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [Delta: table batch reads and writes](https://docs.delta.io/latest/delta-batch.html)
* [DeltaTable.isDeltaTable](https://docs.delta.io/latest/api/python/spark/index.html)
* [Delta: MERGE](https://docs.delta.io/latest/delta-update.html#upsert-into-a-table-using-merge)
* [Change data feed](https://docs.delta.io/latest/delta-change-data-feed.html)
* [functions.regexp_extract](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_extract.html)
* [DataFrame.dropDuplicates](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html)
* [functions.broadcast](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.broadcast.html)
* [functions.coalesce](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.coalesce.html)
* [DeltaTable.history](https://docs.delta.io/latest/delta-utility.html#retrieve-delta-table-history)
* [Delta: optimizations](https://docs.delta.io/latest/optimizations-oss.html)
* [Read the change feed](https://docs.delta.io/latest/delta-change-data-feed.html#read-changes-in-batch-queries)
* [Delta: VACUUM](https://docs.delta.io/latest/delta-utility.html#remove-files-no-longer-referenced-by-a-delta-table)
* [Optimizing skew join](https://spark.apache.org/docs/latest/sql-performance-tuning.html#optimizing-skew-join)
* [DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
* [functions.lag](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.lag.html)
* [functions.datediff](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.datediff.html)

Labs 01 to 09 and their documentation pages are your other reference.

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_customers(spark):
    c = spark.range(1, N_CUSTOMERS + 1).withColumnRenamed("id", "customer_id")
    return c.select(
        "customer_id",
        F.when(F.col("customer_id") == 1, "Big Reseller Ltd")
         .otherwise(F.concat(F.lit("Customer "), F.col("customer_id").cast("string"))).alias("name"),
        _pick(_u(F.col("customer_id"), 1), COUNTRIES).alias("country"),
        F.date_add(F.lit("2022-01-01").cast("date"), F.floor(_u(F.col("customer_id"), 2) * 730).cast("int")).alias("signup_date"),
        F.when((F.col("customer_id") == 1) | (_u(F.col("customer_id"), 3) < 0.2), "business").otherwise("retail").alias("segment"),
    )

def make_products(spark):
    p = spark.range(1, N_PRODUCTS + 1).withColumnRenamed("id", "product_id")
    return p.select(
        "product_id",
        F.concat(F.lit("Product "), F.col("product_id").cast("string")).alias("name"),
        F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (F.col("product_id") % 8 + 1).cast("int")).alias("category"),
        F.round((F.col("product_id") * 37) % 195 + 5.99, 2).alias("price"),
    )

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    customer = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                    # guest checkout: no customer
                 .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                       # the big reseller: 30% of orders
                 .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    quantity = F.when(customer == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), customer.alias("customer_id"), product.alias("product_id"), quantity.alias("quantity"),
        F.round(quantity * (((product * 37) % 195) + 5.99), 2).alias("amount"),
        status.alias("status"), payment.alias("payment_method"), ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "product_id", "quantity", "amount", "status",
                     "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_customers(spark).toPandas().to_csv("data/customers.csv", index=False)
    make_products(spark).toPandas().to_csv("data/products.csv", index=False)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day10_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session(delta=True, driver_memory="3g")
if not all(os.path.exists(f) for f in ("data/orders.parquet", "data/customers.csv", "data/products.csv", "data/monthly/orders_2024-03.parquet")):
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

try:
    from delta.tables import DeltaTable   # for the Delta labs
except ImportError:
    pass

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)

# ---------------------------------------------------------------- checks -----------------
_EXPECTED = {
 "P1": "fdd863e008f03ed3",
 "P2": "461b079fbffc8af6",
 "P3": "5d2de7d06262a18a",
 "P5": "53f02c01609118c1",
 "Q1": "19743fccb02dd73d",
 "Q2": "3a2df1103f1c9343",
 "Q3": "aa17cb782a53bbbe"
}
_RECORD = os.environ.get("LAB_RECORD")

def _norm(v):
    if isinstance(v, bool) or v is None: return v
    if isinstance(v, int): return v
    if isinstance(v, float): return round(v, 2)
    if isinstance(v, str): return v.strip()
    if isinstance(v, dict): return {str(k): _norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
    if isinstance(v, (list, tuple)): return [_norm(x) for x in v]
    if isinstance(v, (set, frozenset)): return sorted((_norm(x) for x in v), key=json.dumps)
    if hasattr(v, "item"): return _norm(v.item())
    if type(v).__name__ == "Decimal": return round(float(v), 2)
    raise TypeError(f"check() needs plain Python values, got {type(v).__name__}")

def check(task, value):
    normal = _norm(value)
    if _RECORD:
        rec = json.load(open(_RECORD)) if os.path.exists(_RECORD) else {}
        rec[task] = normal; json.dump(rec, open(_RECORD, "w"))
    ok = hashlib.sha256(json.dumps(normal).encode()).hexdigest()[:16] == _EXPECTED.get(task)
    print(f"✅ {task} correct" if ok else f"❌ {task} not yet: check your code and the question again")

def check_true(task, condition, hint=""):
    print(f"✅ {task} correct" if condition else f"❌ {task} not yet. {hint}")


Load the tables the way Lab 01 taught you (given):

In [ ]:
orders = spark.read.parquet("data/orders.parquet")
customers = (spark.read.option("header", True)
             .schema("customer_id long, name string, country string, signup_date date, segment string")
             .csv("data/customers.csv"))
products = (spark.read.option("header", True)
            .schema("product_id long, name string, category string, price double")
            .csv("data/products.csv"))

BRONZE, SILVER, QUARANTINE, GOLD = "lake/bronze/orders", "lake/silver/orders", "lake/quarantine/orders", "lake/gold/daily_country"
shutil.rmtree("lake", ignore_errors=True)       # every run of this notebook starts clean

def table(path):
    return spark.read.format("delta").load(path)

## Milestone 1: bronze

**P1. Write `ingest_month(month)` that appends one raw monthly file to the Delta table at `BRONZE`.**
* read `data/monthly/orders_{month}.parquet` as it is and add *source_file* (the file name) and *ingested_at* (the load time)
* it must be **idempotent**: if that file is already in bronze, do nothing
* load January and February (March comes in milestone 5)

Put the number of bronze rows in *bronze_rows*.

In [ ]:
check("P1", bronze_rows)
before = table(BRONZE).count(); ingest_month("2024-01")
check_true("P1b", table(BRONZE).count() == before, "loading January again must not add rows")

## Milestone 2: silver and quarantine

**P2. Write `build_silver()` that turns bronze into the clean Delta table `SILVER` and writes rejected rows to `QUARANTINE`.**
* reject a row, with a *reason* column, by the first rule it breaks, in this order:
  1. `order outside file month`: the year and month of *order_date* differ from the month in *source_file*
  2. `bad quantity`: *quantity* is 0 or less
  3. `negative amount`: *amount* is below 0
* drop exact duplicates of the 10 original columns (everything except *source_file* and *ingested_at*)
* store *amount* as `decimal(10,2)` and add *is_guest* (True when *customer_id* is null)
* load silver with a **MERGE on order_id** (insert only new orders), so running it twice changes nothing
* turn on the **Change Data Feed** for silver from its first version (you need it in milestone 5)

Put the number of silver rows in *silver_rows* and the quarantine counts per reason in *reasons* (`{reason: count}`).

In [ ]:
check("P2", [silver_rows, reasons])
s = table(SILVER)
check_true("P2b", s.select("order_id").distinct().count() == s.count(), "order_id must be unique in silver")
n_before = s.count(); build_silver()
check_true("P2c", table(SILVER).count() == n_before, "a second build_silver() must not add rows")
check_true("P2d", dict(s.dtypes)["amount"].startswith("decimal"), "amount must be decimal")

## Milestone 3: gold

**P3. Write `gold_from(silver_df)` and build the Delta table `GOLD`: one row per *order_date* and *country*.**
* columns: *order_date*, *country*, *orders* (count) and *revenue* (sum of *amount*)
* join *customers* with a **broadcast** join; guest orders have no customer: show them as country `Guest`
* save the silver version you built it from in *gold_built_from* (you need it in milestone 5)

Put the number of gold rows in *gold_rows* and the total revenue (a float, 2 decimals) in *gold_revenue*.

In [ ]:
check("P3", [gold_rows, gold_revenue])
g = table(GOLD)
check_true("P3b", g.agg(F.sum("orders")).first()[0] == table(SILVER).count(), "the orders in gold must add up to the silver row count")
check_true("P3c", "BroadcastHashJoin" in plan(gold_from(table(SILVER)), "simple"), "customers should be broadcast")
check_true("P3d", g.filter("country = 'Guest'").count() > 0, "guest orders should appear as country Guest")

## Milestone 4: performance

**P4. Make lookups of one customer cheap on silver.**
* set `spark.databricks.delta.optimize.maxFileSize` to 4 MB (in bytes) so compaction writes several files
* `OPTIMIZE` silver with `ZORDER BY (customer_id)`
* compare how many files a `customer_id = 42` filter reads (`df.filter(...).inputFiles()`) with all silver files

Put `(files_for_42, all_files)` in *files*.

In [ ]:
check_true("P4", files[1] >= 4 and files[0] < files[1] / 2, "after Z-ordering, the filter should read less than half of the files")
last = DeltaTable.forPath(spark, SILVER).history(1).first()
check_true("P4b", last["operation"] == "OPTIMIZE" and "customer_id" in str(last["operationParameters"]), "the last silver commit should be OPTIMIZE with zOrderBy customer_id")

## Milestone 5: March arrives

**P5. March has arrived. Update gold incrementally, without recomputing it from all of silver.**
* `ingest_month("2024-03")`, then `build_silver()`
* read only silver's **change data feed** since *gold_built_from* (the inserted rows)
* aggregate those rows with `gold_from` and **MERGE** them into gold: add to *orders* and *revenue* for existing (date, country) rows, insert new ones

Put the new number of gold rows in *gold_rows_after* and the total revenue in *gold_revenue_after* (float, 2 decimals).

In [ ]:
check("P5", [gold_rows_after, gold_revenue_after])
from pyspark.testing import assertDataFrameEqual
try:
    assertDataFrameEqual(table(GOLD), gold_from(table(SILVER)))
    same = True
except AssertionError:
    same = False
check_true("P5b", same, "incremental gold must equal gold recomputed from all of silver")

## Milestone 6: your notes (not checked)

**P6. In a markdown cell, write half a page for your team:**
* how many rows each rule rejected, and one example row per reason
* the plan of the gold build: join strategy, number of shuffles, what was pushed down
* what OPTIMIZE and ZORDER changed in `_delta_log`, and what `VACUUM` would delete now
* one thing you would change before running this every month in production

Stretch: force a sort-merge join for gold (broadcast off, AQE off), find the straggler in the Spark UI (customer 1 is the Big Reseller), and fix it with AQE skew handling.

In [ ]:
print("Write your notes in a markdown cell above.")

## Part 2: Timed practice (45 minutes)

Three problems on the *orders*, *customers* and *products* tables, in the style of a coding interview. No hints and no skeleton: set a timer, write each solution from scratch, then run the check. Easy 10 minutes, medium 15, hard 20.

**Q1 (easy, 10 min). In every country, how many customers have at least one *cancelled* order?** Ignore guests. *per_country* = dict `{country: number of customers}`.

In [ ]:
check("Q1", per_country)

**Q2 (medium, 15 min). Return rate over two months.** *january* = the number of customers with at least one *returned* order in January 2024, *both* = how many of them also had a *returned* order in February 2024, *pct* = `both / january` in percent (1 decimal). Put `[january, both, pct]` in *result*.

In [ ]:
check("Q2", result)

**Q3 (hard, 20 min). What was the longest wait between two orders?** For the customers 2 to 6, look at their *paid* orders in time order (*order_ts*, ties by *order_id*) and find the largest number of days between the *order_date* of one order and the next one. *gaps* = list of `(customer_id, largest_gap_in_days)` sorted by *customer_id*.

In [ ]:
check("Q3", gaps)

---
Done? Put the notebook in your own GitHub repository with a short README: it is your first Spark portfolio project. In the Theory part of Day 10, do the mock checkpoint.